In [ ]:
from plot_utils import *
setup_plot_style()

X_JITTER = 1e-6

by_L = load_toric_code_data()
OUT_DIR = FIGPLOT_DIR
os.makedirs(OUT_DIR, exist_ok=True)
print("Found L:", sorted(by_L.keys()))

In [ ]:
# Combined LER row: MWPM / UF / Ours
palette_ler = {
    'mwpm': '#1f77b4',
    'uf': '#ff7f0e',
    'hardware': '#2ca02c',
    'peel': '#d62728',
}

fig, axes = plt.subplots(1, len(L_TARGETS), figsize=(22, 4.8), sharey=False)
for i, L in enumerate(L_TARGETS):
    ax = axes[i]
    data = by_L.get(L)
    if data is None:
        ax.set_title(f"Code Distance={L} (N/A)")
        ax.axis('off')
        continue

    ps: List[float] = data.get('ps', [])
    y_mwpm = data.get('ler', {}).get('mwpm', [])
    y_uf = data.get('ler', {}).get('uf', [])
    y_software = data.get('ler', {}).get('uf_peel_votemax', [])

    if PLOT_AS_SCATTER:
        if y_mwpm and len(y_mwpm) == len(ps):
            xp = [p - X_JITTER for p in ps]
            ax.scatter(xp, y_mwpm, s=SCATTER_SIZE, marker='o', color=palette_ler['mwpm'], edgecolor='black', linewidths=MARKER_EDGE_WIDTH, label='MWPM-based Decoder (Software)')
            ax.plot(ps, y_mwpm, linestyle='-', color=palette_ler['mwpm'], linewidth=2.8)
        if y_uf and len(y_uf) == len(ps):
            xp = ps
            ax.scatter(xp, y_uf, s=SCATTER_SIZE, marker='s', color=palette_ler['uf'], edgecolor='black', linewidths=MARKER_EDGE_WIDTH, label='UF-based Decoder (Software)')
            ax.plot(ps, y_uf, linestyle='-', color=palette_ler['uf'], linewidth=2.8)
        if y_software and len(y_software) == len(ps):
            xp = [p + X_JITTER for p in ps]
            ax.scatter(xp, y_software, s=SCATTER_SIZE, marker='^', color=palette_ler['hardware'], edgecolor='black', linewidths=MARKER_EDGE_WIDTH, label='Ours (Software)')
            ax.plot(ps, y_software, linestyle='-', color=palette_ler['hardware'], linewidth=2.8)
    else:
        if y_mwpm and len(y_mwpm) == len(ps):
            ax.plot(ps, y_mwpm, marker='o', linestyle='-', color=palette_ler['mwpm'], label='MWPM-based Decoder (Software)')
        if y_uf and len(y_uf) == len(ps):
            ax.plot(ps, y_uf, marker='s', linestyle='-', color=palette_ler['uf'], label='UF-based Decoder (Software)')
        if y_software and len(y_software) == len(ps):
            ax.plot(ps, y_software, marker='^', linestyle='-', color=palette_ler['hardware'], label='Ours (Software)')

    ax.set_yscale('log')
    ax.grid(True, axis='y', linestyle='--', linewidth=1.8)
    for gl in ax.get_ygridlines():
        gl.set_linestyle((0, (6, 6)))
        gl.set_linewidth(1.8)
        gl.set_color('#9a9a9a')
        gl.set_alpha(0.65)

    desired_ticks = [p for p in ps if p in (0.001, 0.0015)]
    if desired_ticks:
        ax.set_xticks(desired_ticks)
        ax.set_xticklabels([f"{p:.4f}" for p in desired_ticks], rotation=0, ha='center')

    ax.text(0.95, 0.09, f"Code Distance {L}", transform=ax.transAxes,
            ha='right', va='bottom', fontsize=18)

axes[0].set_ylabel('Logical Error Rate', fontsize=24)
for ax in axes:
    ax.tick_params(axis='x', labelrotation=0)

handles_dict: Dict[str, Any] = {}
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    for handle, label in zip(h, l):
        if label and label not in handles_dict:
            handles_dict[label] = handle

if handles_dict:
    desired_order = ['MWPM-based Decoder (Software)', 'UF-based Decoder (Software)', 'Ours (Software)']
    others = [lbl for lbl in handles_dict.keys() if lbl not in desired_order]
    ordered_labels = [lbl for lbl in desired_order if lbl in handles_dict] + others
    ordered_handles = [handles_dict[lbl] for lbl in ordered_labels]
    leg = fig.legend(ordered_handles, ordered_labels, loc='upper center', ncol=len(ordered_labels), frameon=True, bbox_to_anchor=(0.5, 1.045))
    frame = leg.get_frame()
    frame.set_edgecolor('black')
    frame.set_linewidth(1.8)
    frame.set_facecolor('white')
    frame.set_alpha(1.0)

fig.supxlabel('Physical Error Rate', fontsize=24, y=0.06)
fig.tight_layout(rect=[0, 0.002, 1, 0.98])

save_path = os.path.join(OUT_DIR, 'combined_ler_mwpm_uf_ours_row.pdf')
fig.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.02, format='pdf')
plt.show()
print(f"Saved: {save_path}")
try:
    st = os.stat(save_path)
    print(f"File exists, size={st.st_size} bytes")
except FileNotFoundError:
    print("Save failed: file not found. Current cwd:", os.getcwd())